# 01 — Data Pipeline Validation
Quick visual inspection of all downloaded data. Run after `make data`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from commodity_curve_factors.data.futures_loader import load_front_month_data
from commodity_curve_factors.data.macro_loader import load_macro_data
from commodity_curve_factors.data.inventory_loader import load_inventory_data
from commodity_curve_factors.data.cftc_loader import load_cot_data, compute_net_speculative
from commodity_curve_factors.data.validate import detect_gaps

futures = load_front_month_data()
macro = load_macro_data()
inventory = load_inventory_data()
cot = load_cot_data()

print(f"Front-month: {len(futures)} commodities")
print(f"Macro: {len(macro)} series")
print(f"Inventory: {len(inventory)} series")
print(f"CFTC COT: {len(cot)} rows")

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12), sharex=True)
axes = axes.flatten()

for i, (sym, df) in enumerate(sorted(futures.items())):
    ax = axes[i]
    ax.plot(df.index, df["Close"], linewidth=0.7)
    ax.set_title(sym, fontsize=10)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# Hide unused axes
for j in range(len(futures), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Front-Month Continuous Futures (2005-2024)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (key, label) in zip(axes.flatten(), [
    ("VIX", "VIX"),
    ("DGS10", "10Y Treasury Yield"),
    ("DTWEXBGS", "USD Index"),
    ("T5YIE", "5Y Breakeven Inflation"),
]):
    # macro keys are lowercase file stems (vix, dgs10, usd_index, t5yie)
    lookup = key.lower() if key.lower() in macro else {
        "VIX": "vix", "DGS10": "dgs10", "DTWEXBGS": "usd_index", "T5YIE": "t5yie"
    }.get(key, key.lower())
    if lookup in macro:
        series = macro[lookup]
        col = series.columns[0] if len(series.columns) == 1 else "Close"
        ax.plot(series.index, series[col], linewidth=0.7)
        ax.set_title(label)
    else:
        ax.set_title(f"{label} (NOT FOUND)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (key, label) in zip(axes.flatten(), [
    ("crude_stocks", "Crude Oil Stocks (MBBL)"),
    ("natural_gas_storage", "Natural Gas Storage (BCF)"),
    ("gasoline_stocks", "Gasoline Stocks (MBBL)"),
    ("distillate_stocks", "Distillate Stocks (MBBL)"),
]):
    if key in inventory:
        series = inventory[key]
        col = series.columns[0] if hasattr(series, "columns") and len(series.columns) >= 1 else key
        ax.plot(series.index, series.iloc[:, 0] if hasattr(series, "columns") else series,
                linewidth=0.7)
        ax.set_title(label)
    else:
        ax.set_title(f"{label} (NOT FOUND)")

plt.tight_layout()
plt.show()

In [ ]:
net = compute_net_speculative(cot)

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
for ax, sym in zip(axes.flatten(), ["CL", "NG", "GC", "ZC", "KC", "SI"]):
    if sym in net.columns:
        ax.plot(net.index, net[sym], linewidth=0.7)
        ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
        ax.set_title(f"{sym} Managed Money Net")
    else:
        ax.set_title(f"{sym} (NOT FOUND)")

plt.tight_layout()
plt.show()

In [ ]:
print("=== Gap Detection (>5 consecutive NaN) ===")
for sym, df in sorted(futures.items()):
    gaps = detect_gaps(df, max_consecutive_missing=5, col="Close")
    if gaps:
        print(f"\n{sym}: {len(gaps)} gap(s)")
        for start, end, length in gaps:
            print(f"  {start.date()} -> {end.date()} ({length} days)")
    else:
        print(f"{sym}: no gaps")

In [ ]:
rows = []
for sym, df in sorted(futures.items()):
    rows.append({
        "symbol": sym,
        "start": df.index.min().date(),
        "end": df.index.max().date(),
        "rows": len(df),
        "nan_pct": df["Close"].isna().mean() * 100,
    })

coverage = pd.DataFrame(rows)
print(coverage.to_string(index=False))